# Data Split Notebook

Within this notebook, a core snRNA-seq retina dataset is split between neuronal and non-neuronal cell types and downsampled. The resulting datasets are stored in the same directory as the original under `data_{structure}_ds.h5ad`. If subset, the resulting datasets are under `data_{structure}_ds_{subset}.h5ad`

**Note:** The input data file should be a single-cell RNA-seq dataset in h5ad format, with cell type annotations in the obs dataframe and gene annotations in the var dataframe.

**Note:** If using GitHub for version control and repository sharing, ensure that you add the path to your data folder to the repository's `.gitignore` file, to prevent yourself from exceeding the GitHub's storage limits.

In [ ]:
# libraries
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import celltypist as ct

In [ ]:
# configs
code_folder = "/Users/vbecker/NSForest-ncRNA" # path to the NSForest-ncRNA folder
sys.path.insert(0, os.path.abspath(code_folder))

data_folder = "beckersv_data/" # path to folder containing the input data file (.h5ad format)

to_downsample = True # True if you want to downsample the dataset to a specific number of cells, 
                     # False otherwise

to_downsample_max = 500000 # Maximum number of cells within each strata's final .h5ad file.

seed = 0 # random seed for reproducibility

## Functions

### Discovering downsampling threshold per strata.

With `to_downsample_max` value set in config, we'll generate the ideal `to_downsample_n` for each strata to yield a total number of cells close to `to_downsample_max`.

In [ ]:
def find_threshold(og_anndata, cluster_header, ideal_total):
    """
    Args:
        og_anndata (adata): Anndata object containing the original dataset
        cluster_header (str): Header of the column in og_anndata.obs that contains the cluster labels
        ideal_total (int): Ideal total number of cells to downsample to

    Returns:
        int: The threshold value to use for downsampling the dataset to the ideal total number of cells
    """
    
    # get counts dataframe
    og_counts = pd.DataFrame(og_anndata.obs[cluster_header].value_counts()).reset_index()
    print(og_counts)
    
    # if there's less cells than the ideal total, just return the number of cells
    if og_counts['count'].sum() <= ideal_total:
        return ideal_total

    # set upper and lower limits for binary search
    upper_lim = ideal_total
    lower_lim = int(ideal_total / len(og_counts.index))
    
    # recurse
    return find_threshold_recursive(og_counts, ideal_total, upper_lim, lower_lim)
    
def find_threshold_recursive(count_data, ideal_total, upper_lim, lower_lim):
    """

    Args:
        count_data (pd.DataFrame): Dataframe containing the counts of each cluster
        ideal_total (int): Ideal total number of cells to downsample to
        upper_lim (int): Upper limit for the binary search
        lower_lim (_type_): Lower limit for the binary search

    Returns:
        int: The threshold value to use for downsampling the dataset to the ideal total number of cells
    """
    
    # get middle lim
    mid_thresh = (upper_lim + lower_lim) // 2
    
    # get new total
    new_total = count_data['count'].clip(upper=mid_thresh).sum()
    
    # base case check
    if abs(new_total - ideal_total) <= ideal_total * 0.02:
        return mid_thresh
    
    # binary search exhausted :[
    if upper_lim - lower_lim <= 1:
        return mid_thresh

    # sending the recursive case
    if new_total > ideal_total:
        return find_threshold_recursive(
            count_data,
            ideal_total,
            mid_thresh,
            lower_lim
        )
    else:
        return find_threshold_recursive(
            count_data,
            ideal_total,
            upper_lim,
            mid_thresh
        )
    

### Downsampling

In [ ]:
def downsample(adata, cluster_header, ideal_total, seed, filepath, filename, return_index=False, write_to_file=True):
    """
    Args:
        adata (ad.AnnData): Anndata object containing the dataset to downsample
        cluster_header (str): Header of the column in adata.obs that contains the cluster labels
        ideal_total (int): Ideal total number of cells to downsample to
        seed (int): Random seed for reproducibility
        filepath (str): Path to the folder where the downsampled anndata object will be saved
        filename (str): Name of the file where the downsampled anndata object will be saved
        return_index (bool, optional): Whether to return the indices of the downsampled cells. Defaults to False.
        write_to_file (bool, optional): Whether to write the downsampled anndata object to a .h5ad file. Defaults to True.
        
    Returns:
        np.ndarray: Indices of the downsampled cells if return_index is True, otherwise None
    """
    
    # find threshold
    thresh = find_threshold(adata, cluster_header, ideal_total)
    
    # downsample for all cell types
    idx = ct.samples.downsample_adata(
        adata,
        mode="each",
        by=cluster_header,
        n_cells=thresh,
        random_state=seed,
        return_index=True,
    )
    
    # write to file if requested
    if write_to_file:
        adata[idx, :].to_memory().write_h5ad(filename = filepath + filename)
        
    # return index if requested
    if return_index:
        return idx

## Retina Stratification/Downsample

In [ ]:
# retina-specific configs
majorclass_values = {
    "neuron": ["AC", # amacrine cell
               "BC", # bipolar cell
               "Cone", 
               "HC", # horizontal cell
               "RGC", # retinal ganglion cell
               "Rod"],
    
    "non-neuron": ["Astrocyte",
                   "MG", # Müller glia
                   "Microglia",
                   "RPE"] # retinal pigment epithelium
}

### Loading AnnData file

In [ ]:
file = "data_retina_sn.h5ad"
adata_raw = sc.read_h5ad(file, backed = "r")
adata_raw

### Data exploration

In [ ]:
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

In [ ]:
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

In [ ]:
print(adata_raw.obs["author_cell_type"].value_counts()) # gene counts by cell type

In [ ]:
print(adata_raw.var["feature_type"].value_counts()) # gene counts by feature type

### Defining cluster and strata headers.

In [ ]:
cluster_header = "author_cell_type" # column name in adata.obs that contains the cluster labels
                                    # used in NS-Forest
                                    
strata_header = "majorclass" # column name in adata.obs that contains the strata labels

### Checking cell annotation sizes.

**Note:** Some datasets are too large and need to be downsampled to be run through the pipeline. When downsampling, be sure to have all the granular cluster annotations represented. 

In [ ]:
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

### Compute strata masks.

In [ ]:
# compute neuron mask on the adata_raw object
neuron_mask = adata_raw.obs[strata_header].isin(majorclass_values["neuron"])
neuron_idx = np.flatnonzero(neuron_mask)

# temporary backed view
adata_neuron = adata_raw[neuron_mask, :]

In [ ]:
# compute non-neuron mask on the adata_raw object
nonneuron_mask = adata_raw.obs[strata_header].isin(majorclass_values["non-neuron"])
nonneuron_idx = np.flatnonzero(nonneuron_mask)

# temporary backed view
adata_nonneuron = adata_raw[nonneuron_mask, :]

### Downsampling.

**Note:** Clusters with less cells than specified in `to_downsample_n` will have all cells sampled.

In [ ]:
adata_raw._raw = None # fix an issue with original data that prevented writing

In [ ]:
# downsample for all cell types
downsample(adata_raw, cluster_header, to_downsample_max, seed, data_folder, "data_retina_all_ds.h5ad", return_index=False, write_to_file=True)

In [ ]:
# downsample for neuron cell types
idx = downsample(adata_neuron, cluster_header, to_downsample_max, seed, data_folder, "data_retina_neuron_ds.h5ad", return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = neuron_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = data_folder + "data_retina_ds_neuron.h5ad")

In [ ]:
# downsample for neuron cell types
idx = downsample(adata_neuron, cluster_header, to_downsample_max, seed, data_folder, "data_retina_nonneuron_ds.h5ad", return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = neuron_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = data_folder + "data_retina_ds_neuron.h5ad")